In [566]:
from dataclasses import dataclass

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        movement = self.actions[action]
        next_cell = (cell[0] + movement[0], cell[1] + movement[1])
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell

        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
default_grid

Grid(rows=3, cols=4, step_reward=0, terminals={(0, 3): 1}, walls={(1, 1)})

In [567]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    next_cell, reward = grid.step(cell, action)
                    return reward + gamma * V_old[next_cell[0]][next_cell[1]]
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [568]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
value_iteration converged in 6 iterations


In [569]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                next_cell, reward = grid.step(cell, action)
                return reward + gamma * V[next_cell[0]][next_cell[1]]
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'up': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [570]:
render_policy(default_grid, policy)

 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------


In [571]:
def policy_evaluation(
    grid: Grid, policy, gamma=0.9, theta=1e-6, verbose=False, max_iters=1000
):
    if verbose:
        print('---------- policy_evaluation ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                value = 0
                for action, prob in policy[r][c].items():
                    next_cell, reward = grid.step(cell, action)
                    value += prob * (reward + gamma * V_old[next_cell[0]][next_cell[1]])
                V[r][c] = value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
        i += 1
    converged = delta <= theta
    if converged:
        if verbose:
            print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f'policy_evaluation did not converge in {max_iters} iterations')
    return V, converged


policy_evaluation(default_grid, policy, verbose=True);

---------- policy_evaluation ------------
 0.0  0.0  1.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.0 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0  0.0  0.9 
------------------
0.81  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
policy_evaluation converged in 6 iterations


In [572]:
def policy_iteration(
    grid: Grid,
    policy=None,
    max_iters=1000,
    pass_incumbent_policy=False,
    gamma=0.9,
    theta=1e-6,
    policy_evaluation_verbose=False,
    policy_evaluation_max_iters=1000,
):
    print('---------- policy_iteration ------------')
    if policy is None:
        policy = [[{'up': 1.0} for _ in range(grid.cols)] for _ in range(grid.rows)]
    i = 0
    policy_evaluation_converged = True
    changed = None
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    while changed != 0 and i < max_iters:
        V, policy_evaluation_converged = policy_evaluation(
            grid,
            policy,
            gamma,
            theta,
            verbose=policy_evaluation_verbose,
            max_iters=policy_evaluation_max_iters,
        )
        if not policy_evaluation_converged:
            break
        show_V(grid, V)
        new_policy = read_policy(
            grid, V, gamma, incumbent_policy=policy if pass_incumbent_policy else None
        )
        changed = sum(
            new_policy[r][c] != None
            and (
                max(new_policy[r][c], key=new_policy[r][c].get)
                != max(policy[r][c], key=policy[r][c].get)
            )
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if not policy_evaluation_converged:
        print(
            f"policy_iteration did not converge because policy_evaluation did not converge"
        )
    elif converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [573]:
policy_iteration(default_grid);

---------- policy_iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 4
 ↑  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  ↑  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 1
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
policy_iteration converged in 5 iterations


In [574]:
policy_iteration(default_grid, pass_incumbent_policy=True);

---------- policy_iteration ------------
 0.0  0.0  0.0    0 
 0.0    0  0.0  1.0 
 0.0  0.0  0.0  0.9 
------------------
actions changed = 3
 ↑  ↑  →  1 
 ↑  #  →  ↑ 
 ↑  ↑  →  ↑ 
------------------
 0.0  0.0  1.0    0 
 0.0    0  0.9  1.0 
 0.0  0.0 0.81  0.9 
------------------
actions changed = 2
 ↑  →  →  1 
 ↑  #  →  ↑ 
 ↑  →  →  ↑ 
------------------
 0.0  0.9  1.0    0 
 0.0    0  0.9  1.0 
 0.0 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  →  ↑ 
------------------
policy_iteration converged in 4 iterations


In [575]:
uniform_policy = [
    [
        {a: 1 / len(default_grid.actions) for a in default_grid.actions}
        for _ in range(default_grid.cols)
    ]
    for _ in range(default_grid.rows)
]
uniform_policy

[[{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}],
 [{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}],
 [{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}]]

In [576]:
policy_iteration(
    default_grid,
    policy=uniform_policy,
);

---------- policy_iteration ------------
0.15 0.27 0.51    0 
 0.1    0 0.37 0.52 
 0.1 0.14 0.24 0.31 
------------------
actions changed = 6
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 2
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  ↑  ↑ 
 ↑  →  ↑  ↑ 
------------------
policy_iteration converged in 3 iterations


In [577]:
policy_iteration(
    default_grid,
    policy=uniform_policy,
    pass_incumbent_policy=True
);

---------- policy_iteration ------------
0.15 0.27 0.51    0 
 0.1    0 0.37 0.52 
 0.1 0.14 0.24 0.31 
------------------
actions changed = 6
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  ↑  ↑ 
------------------
0.81  0.9  1.0    0 
0.73    0  0.9  1.0 
0.66 0.73 0.81  0.9 
------------------
actions changed = 0
 →  →  →  1 
 ↑  #  →  ↑ 
 →  →  ↑  ↑ 
------------------
policy_iteration converged in 2 iterations
